# Notebook 1: Data Preparation

Load engineered Telco churn data, apply feature exclusions, compare stratified splits, fit preprocessor on train set, and save artifacts for model training.

In [1]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path('..').resolve()
DATA_PATH = ROOT / 'Telco_customer_churn_engineered.csv'
OUTPUT_DIR = ROOT / 'ml_outputs' / 'splits'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXCLUDE_COLS = {
    'CustomerID', 'Count', 'Country', 'State', 'City', 'Zip Code',
    'Lat Long', 'Latitude', 'Longitude',
    'Churn Label', 'Churn Value', 'Churn Score', 'Churn Reason',
    'risk_score', 'engagement_score', 'tenure_group',
}
TARGET = 'Churn Value'
RANDOM_STATE = 42

In [2]:
df = pd.read_csv(DATA_PATH)
customer_ids = df['CustomerID'].copy()
y = df[TARGET]
X = df.drop(columns=[c for c in EXCLUDE_COLS if c in df.columns])

print(f'Dataset: {df.shape[0]} rows')
print(f'Features: {X.shape[1]} columns')
print(f'Churn rate: {y.mean():.2%}')
X.head()

Dataset: 7043 rows
Features: 43 columns
Churn rate: 26.54%


,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,...,internet_fiber,internet_dsl,no_internet,has_phone,has_multiple_lines,addon_services_count,streaming_services_count,total_services_count,has_streaming_bundle,high_monthly_charge
0,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,...,0,1,0,1,0,2,0,2,0,0
1,Female,No,No,Yes,2,Yes,No,Fiber optic,No,No,...,1,0,0,1,0,0,0,0,0,0
2,Female,No,No,Yes,8,Yes,Yes,Fiber optic,No,No,...,1,0,0,1,1,1,2,3,1,1
3,Female,No,Yes,Yes,28,Yes,Yes,Fiber optic,No,No,...,1,0,0,1,1,2,2,4,1,1
4,Male,No,No,Yes,49,Yes,Yes,Fiber optic,No,Yes,...,1,0,0,1,1,2,2,4,1,1


In [3]:
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = [c for c in X.columns if c not in numeric_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]), numeric_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore')),
        ]), categorical_cols),
    ]
)

print('Numeric:', len(numeric_cols))
print('Categorical:', len(categorical_cols))

Numeric: 27
Categorical: 16


In [4]:
split_candidates = {'70/30': 0.30, '75/25': 0.25, '80/20': 0.20}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
baseline = Pipeline([
    ('prep', preprocessor),
    ('model', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE)),
])

split_results = []
for name, test_size in split_candidates.items():
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=RANDOM_STATE
    )
    scores = cross_val_score(baseline, X_tr, y_tr, cv=cv, scoring='roc_auc', n_jobs=-1)
    split_results.append({
        'split': name, 'test_size': test_size,
        'cv_roc_auc_mean': scores.mean(), 'cv_roc_auc_std': scores.std(),
        'train_rows': len(X_tr), 'test_rows': len(X_te),
    })

split_df = pd.DataFrame(split_results).sort_values('cv_roc_auc_mean', ascending=False)
split_df

,split,test_size,cv_roc_auc_mean,cv_roc_auc_std,train_rows,test_rows
2,80/20,0.20,0.863573,0.011077,5634,1409
0,70/30,0.30,0.862086,0.004859,4930,2113
1,75/25,0.25,0.862028,0.014383,5282,1761


In [5]:
best_split = split_df.iloc[0]
TEST_SIZE = float(best_split['test_size'])
print(f"Selected split: {best_split['split']} (CV ROC-AUC: {best_split['cv_roc_auc_mean']:.4f})")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)
ids_train, ids_test = train_test_split(
    customer_ids, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)

print(f'Train: {len(X_train)} | Test: {len(X_test)}')
print(f'Train churn: {y_train.mean():.2%} | Test churn: {y_test.mean():.2%}')

Selected split: 80/20 (CV ROC-AUC: 0.8636)
Train: 5634 | Test: 1409
Train churn: 26.54% | Test churn: 26.54%


In [6]:
preprocessor.fit(X_train)
joblib.dump(preprocessor, OUTPUT_DIR / 'preprocessor.joblib')

X_train.to_csv(OUTPUT_DIR / 'X_train.csv', index=False)
X_test.to_csv(OUTPUT_DIR / 'X_test.csv', index=False)
y_train.to_csv(OUTPUT_DIR / 'y_train.csv', index=False)
y_test.to_csv(OUTPUT_DIR / 'y_test.csv', index=False)
ids_train.to_csv(OUTPUT_DIR / 'ids_train.csv', index=False)
ids_test.to_csv(OUTPUT_DIR / 'ids_test.csv', index=False)

config = {
    'target': TARGET,
    'test_size': TEST_SIZE,
    'split_label': str(best_split['split']),
    'random_state': RANDOM_STATE,
    'train_rows': int(len(X_train)),
    'test_rows': int(len(X_test)),
    'numeric_cols': numeric_cols,
    'categorical_cols': categorical_cols,
    'exclude_cols': sorted(EXCLUDE_COLS),
}
with open(OUTPUT_DIR / 'feature_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print('Artifacts saved to', OUTPUT_DIR)

Artifacts saved to C:\Users\DELL\cursor demo\customer-churn\ml_outputs\splits
